In [364]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join('.', '..', 'driver')))
import importlib
import kinematics
importlib.reload(kinematics)
from kinematics import azaltroll_to_theta,apply_mechanical_corrections, q_from_azaltroll, MountModelParams


In [365]:
#csv_filename = './n92_validate.csv'
#csv_filename = './n196_validate.csv'
#csv_filename = './n442_validate.csv'
#csv_filename = './n452_validate.csv'
#csv_filename = './n893_validate.csv'
#csv_filename = './n1780_validate.csv'
csv_filename = './n1780b_validate.csv'
d = pd.read_csv(csv_filename)
d.describe()
d['dev_p_theta1'] = ((d['s_theta1'] - d['p_theta1'] + 180) % 360 - 180)*60
d['dev_p_theta2'] = ((d['s_theta2'] - d['p_theta2'] + 180) % 360 - 180)*60
d['dev_p_theta3'] = ((d['s_theta3'] - d['p_theta3'] + 180) % 360 - 180)*60
if 'q_theta1' in d.columns:
    d['dev_q_theta1'] = ((d['s_theta1'] - d['q_theta1'] + 180) % 360 - 180)*60
    d['dev_q_theta2'] = ((d['s_theta2'] - d['q_theta2'] + 180) % 360 - 180)*60
    d['dev_q_theta3'] = ((d['s_theta3'] - d['q_theta3'] + 180) % 360 - 180)*60
if 'm_theta1' in d.columns:
    d['dev_m_theta1'] = ((d['s_theta1'] - d['m_theta1'] + 180) % 360 - 180)*60
    d['dev_m_theta2'] = ((d['s_theta2'] - d['m_theta2'] + 180) % 360 - 180)*60
    d['dev_m_theta3'] = ((d['s_theta3'] - d['m_theta3'] + 180) % 360 - 180)*60
d['g_az'] = np.round(d['p_az'] / 5) * 5 
d['g_alt'] = np.round(d['p_alt'] / 5) * 5
d['g_roll'] = np.round(d['p_roll'] / 5) * 5
d.columns

Index(['session_id', 'filename', 'status', 'date_obs', 'site_lat', 'site_lon',
       'p_az', 'p_alt', 'p_roll', 'p_theta1', 'p_theta2', 'p_theta3', 's_az',
       's_alt', 's_roll', 's_theta1', 's_theta2', 's_theta3', 'dev_p_az',
       'dev_p_alt', 'dev_p_roll', 'pixel_scale_arcsec', 'sync_point',
       'alignQ_w', 'alignQ_x', 'alignQ_y', 'alignQ_z', 'q_az', 'q_alt',
       'q_roll', 'q_theta1', 'q_theta2', 'q_theta3', 'dev_q_az', 'dev_q_alt',
       'dev_q_roll', 'm_az', 'm_alt', 'm_roll', 'm_theta1', 'm_theta2',
       'm_theta3', 'dev_m_az', 'dev_m_alt', 'dev_m_roll', 'dev_p_theta1',
       'dev_p_theta2', 'dev_p_theta3', 'dev_q_theta1', 'dev_q_theta2',
       'dev_q_theta3', 'dev_m_theta1', 'dev_m_theta2', 'dev_m_theta3', 'g_az',
       'g_alt', 'g_roll'],
      dtype='object')

In [366]:
if 'm_theta1' in d.columns:
    d[['m_az', 'm_alt', 'm_roll', 'm_theta1', 'm_theta2', 'm_theta3','dev_m_theta1','dev_m_theta2','dev_m_theta3'  ]].corr(numeric_only=True)

# Review of Collected Data and Residuals

In [367]:
fig = go.Figure()
legend_offset = 400
d['legend_az'] = d['p_az']/360*100 + legend_offset
d['legend_alt'] = d['p_alt'] + legend_offset
d['legend_roll'] = d['p_roll'] + legend_offset
xfield='date_obs'
yfields = [
    'dev_p_az','dev_p_alt','dev_p_roll',
    'dev_q_theta1','dev_p_theta2','dev_p_theta3',
    'legend_az','legend_alt','legend_roll'
]
for i, yfield in enumerate(yfields):
    hoverextra = "File: %{customdata[0]}<br>" if i==0 else ''
    fig.add_trace(go.Scatter(
        x=d[xfield], y=d[yfield], customdata=d[['filename']],
        mode='markers+lines', name=yfield,
        marker=dict(size=6),
        hovertemplate=f"{hoverextra}{yfield}: %{{y}}<extra></extra>"
    ))

fig.update_layout(
    title=dict(text=f'Axis Residuals vs {xfield}', x=0.5, font=dict(size=24, family='Arial')),
    xaxis_title=f'{xfield}', yaxis_title=f'Axis Residuals (arc-min)', 
    plot_bgcolor='rgba(200, 200, 250, 0.5)',
    hovermode='x unified',
    height=800, width=1400
)

fig.show()

# Theta Space Residuals - BEFORE

In [368]:
fig = px.scatter_matrix(d, dimensions=["dev_p_theta1", "dev_p_theta2", "dev_p_theta3", "p_theta1", "p_theta2", "p_theta3"], color="p_az",
                        hover_data=[ "p_az", "p_alt", "p_roll", "filename",])
fig.update_layout(height=1000,width=1000 )
fig.show()

# Theta Space Residuals - QUEST ONLY

In [369]:
fig = px.scatter_matrix(d, dimensions=["dev_q_theta1", "dev_q_theta2", "dev_q_theta3", "q_theta1", "q_theta2", "q_theta3"], color="q_az",
                        hover_data=[ "p_az", "p_alt", "p_roll", "filename",])
fig.update_layout(height=1000,width=1000 )
fig.show()

# Theta Space Residuals - MODEL + QUEST

In [370]:
fig = px.scatter_matrix(d, dimensions=["dev_m_theta1","dev_m_theta2", "dev_m_theta3", "m_theta1", "m_theta2", "m_theta3"], color="m_alt",
                        hover_data=[ "p_az", "p_alt", "p_roll", "filename",])
fig.update_layout(height=1000,width=1000 )
fig.show()

# Sky Space Residuals - BEFORE

In [371]:
fig = px.scatter_matrix(d, dimensions=["dev_p_az", "dev_p_alt", "dev_p_roll", "p_theta1", "p_theta2", "p_theta3"], color="p_alt",
                        hover_data=[ "p_az", "p_alt", "p_roll", "filename",])
fig.update_layout(height=1000,width=1000 )
fig.show()

# Sky Space Residuals - QUEST ONLY

In [372]:
fig = px.scatter_matrix(d, dimensions=["dev_q_az", "dev_q_alt", "dev_q_roll", "q_theta1", "q_theta2", "q_theta3"], color="q_theta3",
                        hover_data=[ "p_az", "p_alt", "p_roll", "filename",])
fig.update_layout(height=1000,width=1000 )
fig.show()

# Sky Space Residuals - MODEL + QUEST

In [373]:
fig = px.scatter_matrix(d, dimensions=["dev_m_az", "dev_m_alt", "dev_m_roll", "m_theta1", "m_theta2", "m_theta3"], color="m_theta3",
                        hover_data=[ "p_az", "p_alt", "p_roll", "filename",])
fig.update_layout(height=1000,width=1000 )
fig.show()

# Sky Space Residuals vs Theta Space Residuals

In [374]:
fig = px.scatter_matrix(d, dimensions=["dev_m_az", "dev_m_alt", "dev_m_roll", "dev_m_theta1", "dev_m_theta2", "dev_m_theta3"], color="m_theta2")
fig.update_layout(height=1000,width=1000 )
fig.show()

#  VII. Mechnical Correction (arcmin) by Roll and Altitude

In [375]:
params = MountModelParams.from_config({
    "m3_tilt_alt": -130.143651,
    "m3_tilt_az": 96.615292,
    "m3_tilt_bore": 124.284942,
    "m2_tilt_alt_amp": 66.68476,
    "m2_tilt_alt_zero": 0.272734,
    "m3_encoder_scale": 0,
    "m2_roll_coupling": -0.369607,
    "m2_roll_zero": 58.456701,
    "m1_offset": 0.0,
    "m2_offset": 0.0,
    "m3_offset": 0.0
})

altrange = range(70, -1, -10)
rollrange = range(-70, +71, 10)
h = f"              Roll | "
j = f"-------------------|"
for roll in rollrange:
    h = h + f"{roll:+5.0f}° | "
    j = j + f"--------|"
print(h)
print(j)
for alt in altrange:
    s = f"**Altitude {alt:2.0f}°**   | "
    for roll in rollrange:
        q = q_from_azaltroll(180,alt,roll)
        q_adj, mag = apply_mechanical_corrections(q,params)
        mag = mag*60
        s = s + f"{mag:5.0f}' | "
    print(s)


              Roll |   -70° |   -60° |   -50° |   -40° |   -30° |   -20° |   -10° |    +0° |   +10° |   +20° |   +30° |   +40° |   +50° |   +60° |   +70° | 
-------------------|--------|--------|--------|--------|--------|--------|--------|--------|--------|--------|--------|--------|--------|--------|--------|
**Altitude 70°**   |   773' |  2207' |  1960' |  1647' |  1277' |   860' |   408' |    63' |   533' |   986' |  1405' |  1776' |  2090' |  2339' |   661' | 
**Altitude 60°**   |  3630' |  3384' |  3036' |  2585' |  2034' |  1394' |   687' |    58' |   803' |  1512' |  2154' |  2708' |  3162' |  3513' |  3762' | 
**Altitude 50°**   |  4893' |  4606' |  4187' |  3623' |  2904' |  2030' |  1028' |    51' |  1131' |  2136' |  3014' |  3739' |  4309' |  4733' |  5023' | 
**Altitude 40°**   |  6178' |  5883' |  5437' |  4806' |  3950' |  2839' |  1479' |    43' |  1566' |  2931' |  4050' |  4914' |  5553' |  6007' |  6307' | 
**Altitude 30°**   |  7486' |  7220' |  6803' |  6178' |  5

# Scrap Area


In [376]:
fig = px.scatter(d, x="dev_q_theta2", y="dev_q_theta3", color="q_theta2", hover_data=[ "p_az", "p_alt", "p_roll", "filename",])
fig.update_layout(height=500,width=500 )
fig.show()

# Fit dev_q_theta2 = intercept + m3_tilt_alt * q_theta3

In [377]:
fig = px.scatter(d, x="q_theta3", y="dev_q_theta2", color="q_az", trendline="ols",
                 hover_data=[ "p_az", "p_alt", "p_roll", "filename",])
fig.update_layout(height=500,width=800 )
results = px.get_trendline_results(fig)
model = results.iloc[0]["px_fit_results"]
c = model.params[0]   # intercept
m = model.params[1]   # slope
r2 = model.rsquared
eqn_text = f"y = {c:.3f}  {m:+.3f} x   (R²={r2:.3f})"
fig.add_annotation(x=0.95, y=0.95, xref="paper", yref="paper", text=eqn_text, showarrow=False, font=dict(size=14))
fig.show()

In [378]:
# Fit dev_theta2 = intercept + m3_tilt_z * sin_theta2 * theta3

In [379]:
d["indep"] = 1 - np.cos(np.radians(d["q_theta3"]))
fig = px.scatter(d, x="indep", y="dev_q_theta1", color="q_alt", trendline="ols",
                 hover_data=[ "p_az", "p_alt", "p_roll", "filename",])
fig.update_layout(height=500,width=800 )
results = px.get_trendline_results(fig)
model = results.iloc[0]["px_fit_results"]
c = model.params[0]   # intercept
m = model.params[1]   # slope
r2 = model.rsquared
eqn_text = f"y = {c:.3f}  {m:+.3f} x   (R²={r2:.3f})"
fig.add_annotation(x=0.95, y=0.95, xref="paper", yref="paper", text=eqn_text, showarrow=False, font=dict(size=14))
fig.show()

In [380]:
d["indep"] = 1 - np.cos(np.radians(d["q_theta3"]))
fig = px.scatter(d, x="indep", y="dev_q_theta3", color="q_alt", trendline="ols",
                 hover_data=[ "p_az", "p_alt", "p_roll", "filename",])
fig.update_layout(height=500,width=800 )
results = px.get_trendline_results(fig)
model = results.iloc[0]["px_fit_results"]
c = model.params[0]   # intercept
m = model.params[1]   # slope
r2 = model.rsquared
eqn_text = f"y = {c:.3f}  {m:+.3f} x   (R²={r2:.3f})"
fig.add_annotation(x=0.05, y=0.95, xref="paper", yref="paper", text=eqn_text, showarrow=False, font=dict(size=14))
fig.show()

# Confirm dev_t3 = -cos(theta2) * dev_t1

In [384]:
d["indep"] = np.cos(np.radians(d["q_theta2"]))*d["dev_q_theta1"]
fig = px.scatter(d, x="indep", y="dev_q_theta3", color="q_alt", trendline="ols",
                 hover_data=[ "p_az", "p_alt", "p_roll", "filename",])
fig.update_layout(height=500,width=800 )
results = px.get_trendline_results(fig)
model = results.iloc[0]["px_fit_results"]
c = model.params[0]   # intercept
m = model.params[1]   # slope
r2 = model.rsquared
eqn_text = f"y = {c:.3f}  {m:+.3f} x   (R²={r2:.3f})"
fig.add_annotation(x=0.95, y=0.95, xref="paper", yref="paper", text=eqn_text, showarrow=False, font=dict(size=14))
fig.show()

In [382]:
fig = px.scatter_matrix(d, dimensions=["dev_m_theta2", "m_theta1", "m_theta2", "m_theta3"], color="m_az")
fig.update_layout(height=1000,width=1000 )
fig.show()